# One-stage anchor-free detector: YOLOv8
Setup


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches

In [ ]:
import os
from pathlib import Path
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('kaggle')

if not Path('/content/data/train').exists():
    !kaggle competitions download -c multiple-object-video-detection-hw-2-ucu-ai-2026 -p /content/data
    !cd /content/data && unzip -q '*.zip' && rm *.zip
else:
    print('Data is here')
ROOT = Path('/content/data')

100% 9.88G/9.88G [09:03<00:00, 19.5MB/s]



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PREP = Path('/content/prepared')
if not PREP.exists():
    !unzip -q /content/drive/MyDrive/УКУ/AI/hw2_prepared.zip -d /content
for split in ['train', 'val']:
    n = sum(1 for _ in (PREP / split / 'images').rglob('*.jpg'))
    print(f'{split}: {n} кадрів')

Mounted at /content/drive
train: 4868 кадрів
val: 571 кадрів


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics]


## Data and training protocol
Ultralytics reads the prepared (masked) frames and their YOLO labels directly through `visdrone8.yaml`. All runs use the same budget as the other detectors: 6 epochs, input 1280 (the frames are 1280×720), COCO-pretrained YOLOv8s weights, and horizontal flip as the only augmentation (`FLIP_ONLY`).

In [ ]:
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), 'No GPU: Runtime → Change runtime type → T4'

yaml_path = PREP / 'visdrone8.yaml'
yaml_path.write_text(f"""
path: {PREP}
train: train/images
val: val/images
names: {{0: person, 1: bicycle, 2: car, 3: van, 4: truck, 5: tricycle, 6: bus, 7: motor}}
""")

RUNS = '/content/drive/MyDrive/УКУ/AI/hw2_runs_yolo'      # куди зберігати ваги (твоя папка на Drive)
WEIGHTS = '/content/drive/MyDrive/УКУ/AI/yolov8s.pt'      # pretrained-ваги

PROTOCOL = dict(data=str(yaml_path), imgsz=1280, epochs=6, batch=4, nbs=4, seed=42,
                optimizer='SGD', lr0=0.005, lrf=0.1, momentum=0.9, weight_decay=1e-4,
                warmup_epochs=0.25, amp=True, device=0, cache='disk',
                workers=8, val=False, plots=False, deterministic=False,
                project=RUNS, exist_ok=True)
FLIP_ONLY = dict(fliplr=0.5, mosaic=0, hsv_h=0, hsv_s=0, hsv_v=0, translate=0, scale=0)

### Initial model: YOLOv8s with the SGD protocol
SGD, lr 0.005, batch 4, no gradient accumulation (`nbs=4`), as in the anchor-based notebook.

In [ ]:
model = YOLO(WEIGHTS)
model.train(**PROTOCOL, **FLIP_ONLY, name='v8s_baseline')

Ultralytics 8.4.214 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/prepared/visdrone8.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0, hsv_s=0, hsv_v=0, imgsz=1280, iou=None, kobj=1.0, line_width=None, lr0=0.005, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/УКУ/AI/yolov8s.pt, momentum=0.9, mosaic=0, multi_scale=0.0, name=v8s_baseline, nbs=4, nms=None, opset=N

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b279a13b250>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

### Same model with Ultralytics' automatic optimizer
`optimizer='auto'` selects AdamW (lr 8.3e-4). `nbs` stays at its default of 64, so gradients are accumulated over 4 batches of 16: about 76 optimizer updates per epoch.

In [ ]:
PROTOCOL_AUTO = dict(data=str(yaml_path), imgsz=1280, epochs=6, batch=16, seed=42,
                     optimizer='auto',        # Ultralytics сам обирає оптимізатор і lr
                     warmup_epochs=1.0,       # default 3.0 було б половиною з 6 епох
                     amp=True, device=0, cache='disk',
                     workers=8, val=False, plots=False, deterministic=False,
                     project=RUNS, exist_ok=True)

YOLO(WEIGHTS).train(**PROTOCOL_AUTO, **FLIP_ONLY, name='v8s_baseline_auto')

Ultralytics 8.4.214 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/prepared/visdrone8.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0, hsv_s=0, hsv_v=0, imgsz=1280, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/УКУ/AI/yolov8s.pt, momentum=0.937, mosaic=0, multi_scale=0.0, name=v8s_baseline_auto, nbs=64, nms=None

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b277fb30640>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

### P2 head, first attempt
`yolov8s-p2.yaml` adds a stride-4 detection head. `.load(WEIGHTS)` matches layers by index, so only 219 of 437 tensors are initialised from the pretrained model (see the log); the bottom-up neck and the detection head start from random weights. The run also fell back to batch 8 after an out-of-memory error in the label assigner (609 iterations per epoch instead of 305).

In [ ]:
model_p2 = YOLO('yolov8s-p2.yaml').load(WEIGHTS)        # та сама YOLOv8s + четверта голова на P2
print('strides:', model_p2.model.model[-1].stride)       # має бути [4, 8, 16, 32]
model_p2.train(**PROTOCOL_AUTO, **FLIP_ONLY, name='v8s_p2_auto')

Transferred 219/437 items from pretrained weights
strides: tensor([ 4.,  8., 16., 32.])
Ultralytics 8.4.214 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/prepared/visdrone8.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0, hsv_s=0, hsv_v=0, imgsz=1280, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-p2.yaml, momentum=0.937, mosa

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b277ef9b530>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

## Evaluation
Same as for the torchvision models: predictions on the original val frames (every 5th frame, 571, not masked), pycocotools with `maxDets=500`, ignored regions handled by `iscrowd=1`. The metrics printed by Ultralytics during training are computed on the masked val set without ignored regions, so they are not used for comparison.

In [ ]:
import json, contextlib, io, time, numpy as np, pandas as pd, torch
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO

CLASSES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']
SEED, IMGSZ = 42, 1280

with contextlib.redirect_stdout(io.StringIO()):
    gt = COCO(str(ROOT / 'val/annotations_coco.json'))
imgs_va = pd.DataFrame(gt.dataset['images']).rename(columns={'id': 'image_id'})
val_sub = imgs_va.sort_values(['video', 'frame_id'])
val_sub = val_sub[val_sub.groupby('video').cumcount() % 5 == 0]
print('val frames for evaluation:', len(val_sub))


def evaluate(preds, img_ids):
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt.loadRes(preds)
        E = COCOeval(gt, dt, 'bbox')
        E.params.imgIds = list(img_ids); E.params.maxDets = [1, 100, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats
    res = {'mAP': s[0], 'AP50': s[1], 'AP75': s[2], 'AP_small': s[3], 'AP_medium': s[4], 'AP_large': s[5]}
    prec = E.eval['precision']
    for k, c in enumerate(CLASSES):
        p = prec[:, :, k, 0, -1]
        res[f'AP_{c}'] = float(p[p > -1].mean()) if (p > -1).any() else float('nan')
    return res


def predict_yolo(model, df, split='val', imgsz=IMGSZ, bs=16):
    files = [str(ROOT / split / f) for f in df.file_name]
    ids, out = df.image_id.tolist(), []
    for i in range(0, len(files), bs):
        res = model.predict(files[i:i + bs], imgsz=imgsz, conf=0.001, max_det=300, half=True, device=0, verbose=False)
        for iid, r in zip(ids[i:i + bs], res):
            b = r.boxes.xyxy.float().cpu().numpy().copy(); b[:, 2:] -= b[:, :2]       # xyxy -> xywh
            for box, s, c in zip(b, r.boxes.conf.float().cpu().numpy(), r.boxes.cls.cpu().numpy()):
                out.append({'image_id': int(iid), 'category_id': int(c) + 1,           # 0..7 -> 1..8
                            'bbox': [round(float(v), 2) for v in box], 'score': float(s)})
    return out


def measure_speed_yolo(model, df, n=50, warmup=5, imgsz=IMGSZ):
    """Batch 1; inference + NMS (as torchvision models, whose forward includes NMS); image reading excluded."""
    times = []
    for i, f in enumerate(df.file_name.iloc[:n]):
        r = model.predict(str(ROOT / 'val' / f), imgsz=imgsz, conf=0.001, max_det=300,
                          half=True, device=0, verbose=False)[0]
        if i >= warmup:
            times.append(r.speed['inference'] + r.speed['postprocess'])
    ms = float(np.mean(times))
    return {'ms_per_img': round(ms, 1), 'FPS': round(1000 / ms, 1)}


def eval_run(weights, imgsz=IMGSZ):
    m = YOLO(weights)
    return {**evaluate(predict_yolo(m, val_sub, imgsz=imgsz), val_sub.image_id), **measure_speed_yolo(m, val_sub, imgsz=imgsz)}

val frames for evaluation: 571


In [ ]:
p2_yolo = YOLO(f'{RUNS}/v8s_p2_auto/weights/last.pt')
res_p2 = evaluate(predict_yolo(p2_yolo, val_sub), val_sub.image_id)
pd.Series(res_p2)

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

,0
mAP,0.161037
AP50,0.344712
AP75,0.126377
AP_small,0.080318
AP_medium,0.156082
AP_large,0.282939
AP_person,0.258282
AP_bicycle,0.143265
AP_car,0.350508
AP_van,0.122463


In [ ]:
from IPython.display import display
p2_yolo = YOLO("/content/drive/MyDrive/УКУ/AI/hw2_runs_yolo/v8s_p2_auto/weights/best.pt")
p2_yolo = evaluate(predict_yolo(p2_yolo, val_sub, imgsz=1280), val_sub.image_id)
display(pd.Series(p2_yolo))
yolo = YOLO ('/content/drive/MyDrive/УКУ/AI/hw2_runs_yolo/v8s_baseline/weights/best.pt')
res_yolo = evaluate(predict_yolo(yolo, val_sub, imgsz=1280), val_sub.image_id)
display(pd.Series(res_yolo))

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

,0
mAP,0.161037
AP50,0.344712
AP75,0.126377
AP_small,0.080318
AP_medium,0.156082
AP_large,0.282939
AP_person,0.258282
AP_bicycle,0.143265
AP_car,0.350508
AP_van,0.122463


WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

,0
mAP,0.160578
AP50,0.328230
AP75,0.134297
AP_small,0.088033
AP_medium,0.165637
AP_large,0.258564
AP_person,0.252778
AP_bicycle,0.138019
AP_car,0.361420
AP_van,0.124825


## Tuned experiments
The first P2 run is not a fair test of the extra head: half of the network was randomly initialised, and its batch size differed from the baseline. The runs below fix both.
- `build_p2` copies every layer that also exists in plain YOLOv8s (including the P3–P5 head branches) into the P2 model, so only the new stride-4 branch starts from random weights (437/437 tensors transferred at training start, see the log).
- `PROTOCOL_V2`: AdamW, lr 5e-4, effective batch 8 without accumulation (609 updates per epoch), warm-up from zero for all parameter groups, validation after every epoch.

Three runs: a control (plain YOLOv8s), experiment 1 (P2 head) and experiment 2 (P2 head with input 1600).

In [ ]:
import copy, torch
from ultralytics import YOLO
from ultralytics.nn.modules import Conv


def build_p2(weights, cfg='yolov8s-p2.yaml'):
    """YOLOv8s-P2 with every layer that also exists in plain YOLOv8s initialised from the pretrained weights."""
    src = YOLO(weights).model                      # plain YOLOv8s: layers 0-21 + Detect(P3, P4, P5)
    m = YOLO(cfg).load(weights)                    # by index: backbone and the top-down path (layers 0-15)
    dst = m.model
    for d, s in {22: 16, 24: 18, 25: 19, 27: 21}.items():        # bottom-up path sits 6 layers later in the P2 graph
        dst.model[d].load_state_dict(src.model[s].state_dict())
    hd, hs = dst.model[-1], src.model[-1]
    c3 = hs.cv3[0][-1].in_channels                 # width of the pretrained cls branch
    for i in range(3):                             # P3, P4, P5 branches -> positions 1, 2, 3 of the P2 head
        hd.cv2[i + 1].load_state_dict(hs.cv2[i].state_dict())
        hd.cv3[i + 1] = copy.deepcopy(hs.cv3[i])
    p2_in = hd.cv2[0][0].conv.in_channels          # new P2 cls branch at the same width (random init)
    hd.cv3[0] = torch.nn.Sequential(Conv(p2_in, c3, 3), Conv(c3, c3, 3), torch.nn.Conv2d(c3, hs.nc, 1))
    hd.bias_init()
    for i in range(3):                             # bias_init reset the copied output biases: copy them again
        hd.cv2[i + 1][-1].load_state_dict(hs.cv2[i][-1].state_dict())
        hd.cv3[i + 1][-1].load_state_dict(hs.cv3[i][-1].state_dict())
    dst.names = src.names                          # lets the trainer keep the COCO rows of shared classes
    return m


PROTOCOL_V2 = dict(data=str(yaml_path), epochs=6, seed=42,
                   optimizer='AdamW', lr0=5e-4, momentum=0.9, weight_decay=5e-4,
                   nbs=8,                          # effective batch 8: no accumulation to 64
                   warmup_epochs=1.0, warmup_bias_lr=0.0,   # 0.1 is an SGD default and wrecks AdamW fine-tuning
                   amp=True, device=0, cache='disk', workers=8,
                   val=True, plots=False, deterministic=False, project=RUNS, exist_ok=True)

In [ ]:
# control: plain YOLOv8s, corrected protocol
YOLO(WEIGHTS).train(**PROTOCOL_V2, **FLIP_ONLY, imgsz=1280, batch=8, name='v8s_v2')
# experiment 1: P2 head, properly initialised
build_p2(WEIGHTS).train(**PROTOCOL_V2, **FLIP_ONLY, imgsz=1280, batch=8, name='v8s_p2_v2')
# experiment 2: + input resolution 1600 (batch 4, still effective batch 8)
build_p2(WEIGHTS).train(**PROTOCOL_V2, **FLIP_ONLY, imgsz=1600, batch=4, name='v8s_p2_1600')

Ultralytics 8.4.214 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/prepared/visdrone8.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=6, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0, hsv_s=0, hsv_v=0, imgsz=1280, iou=None, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/УКУ/AI/yolov8s.pt, momentum=0.9, mosaic=0, multi_scale=0.0, name=v8s_v2, nbs=8, nms=None, opset=None,

TypeError: ERROR ❌️ /content/drive/MyDrive/УКУ/AI/hw2_runs_yolo/v8s_p2_1600/weights/best.pt is not a loadable checkpoint — the file is empty, truncated or corrupted (RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory. This is an internal miniz error. If you are seeing this error, there is a high likelihood that your checkpoint file is corrupted. This can happen if the checkpoint was not saved properly, was transferred incorrectly, or the file was modified after saving.).
Recommended fixes are to re-download or re-export the file, or to run a command with an official Ultralytics model, i.e. 'yolo predict model=yolo26n.pt'

The log above ends with an error: after the third run finished its 6 epochs, the automatic final validation could not read `best.pt`, which was truncated on Google Drive because the Drive quota was full. `last.pt` of that run was saved correctly and is used in the table below.

## Results: initial and tuned models, by object size
Every run is scored with the same `evaluate` on the same 571 original val frames, using the weights of the last epoch.

In [ ]:
from IPython.display import display
RUN_LIST = [('v8s_baseline',      1280, 'initial: SGD lr 0.005, batch 4'),
            ('v8s_baseline_auto', 1280, 'AdamW auto, effective batch 64'),
            ('v8s_p2_auto',       1280, 'P2 head, weights loaded by index'),
            ('v8s_v2',            1280, 'control: AdamW 5e-4, effective batch 8'),
            ('v8s_p2_v2',         1280, 'P2 head, remapped pretrained weights'),
            ('v8s_p2_1600',       1600, 'P2 head + input 1600')]
rows = {}
for name, sz, note in RUN_LIST:
    try:
        rows[name] = {**eval_run(f'{RUNS}/{name}/weights/last.pt', imgsz=sz), 'setting': note}
    except Exception as e:
        print(name, 'skipped:', str(e)[:80])

# reference rows: final results of the anchor-based detectors, saved by the anchor-based notebook
for name in ['frcnn_baseline', 'frcnn_exp1_roi_sampling', 'frcnn_exp2_giou']:
    p = Path('/content/drive/MyDrive/hw2_runs') / f'{name}_hist.json'
    if p.exists():
        rows[name] = {**json.load(open(p))[-1], 'setting': 'anchor-based reference'}

table = pd.DataFrame(rows).T
metrics = ['mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large']
display(pd.concat([table[['setting']], table[metrics].astype(float).round(4)], axis=1))

## How locations are assigned without anchors
YOLOv8 has no anchor boxes: every feature-map point (strides 8, 16, 32; plus 4 with the P2 head) predicts the distances to the four box sides as a distribution (DFL) and a class score. The task-aligned assigner makes a point positive for an object if it lies inside the box and is among the top 10 points by score^0.5 × IoU^6; a point claimed by several objects goes to the one with the highest IoU. One object therefore gets several positives, and NMS is still needed at inference.

**Small objects:** a 10 px object covers about one stride-8 point, so it gets very few positives (Ultralytics enlarges tiny boxes to the stride size when selecting candidates). A stride-4 head puts about four times more points inside the same box. **Dense scenes:** neighbouring objects compete for the same points, and overlapping same-class boxes can be removed by NMS.

## Search range and selected settings
| Parameter | Values tried | Selected |
|---|---|---|
| Optimisation recipe | SGD lr 0.005, batch 4; AdamW auto (lr 8.3e-4, effective batch 64); AdamW 5e-4, effective batch 8 | TODO: fill from the table |
| Detection head | P3–P5; P2–P5 | P2–P5 |
| P2 initialisation | weights matched by layer index (219/437 tensors); remapped (437/437) | remapped |
| Input resolution | 1280; 1600 | TODO: fill from the table |

## Discussion
The per-epoch numbers below are Ultralytics' validation mAP50-95 from the training logs (masked val frames); the table above is the comparison with the common metric.
- The P2 head helps only when it is properly initialised: under the same protocol the control reached 0.174 and the P2 model 0.189. With index-matched weights the P2 model had scored 0.168.
- Raising the input to 1600 gave no reliable gain: 0.184 after 6 epochs (0.194 at epoch 5).
- The plain model peaks at epoch 2 (0.179) and then declines to 0.174: with flip-only augmentation it overfits the training videos.
- TODO after running the table: compare with the anchor-based rows for small, medium and large objects.